# ML Market Prediction - Complete Tutorial

This notebook demonstrates the complete ML infrastructure for predicting market direction (UP/DOWN).

## Table of Contents
1. Setup & Installation
2. Feature Engineering
3. Model Training
4. Making Predictions
5. Performance Analysis
6. Advanced Usage

## 1. Setup & Installation

In [ ]:
# Install dependencies (run once)
!pip install -r ML/requirements_ml.txt

In [ ]:
# Import ML modules
import sys
sys.path.append('..')  # Add parent directory to path

from ML import MLEngine, MLTrainer, MLPredictor, FeatureEngineer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Imports successful!")

## 2. Feature Engineering

Extract 50+ technical indicators from market data

In [ ]:
# Initialize feature engineer
engineer = FeatureEngineer()

# Extract features for SPY (S&P 500)
symbol = 'SPY'
X, y = engineer.prepare_training_data(symbol, lookback_days=365)

print(f"📊 Dataset Shape: {X.shape}")
print(f"📈 Features: {X.shape[1]}")
print(f"📉 Samples: {X.shape[0]}")
print(f"\n🎯 Target Distribution:")
print(f"   UP (1):   {sum(y)} ({sum(y)/len(y)*100:.1f}%)")
print(f"   DOWN (0): {len(y)-sum(y)} ({(len(y)-sum(y))/len(y)*100:.1f}%)")

In [ ]:
# Display feature names
print("📋 Feature Names:")
for i, feature in enumerate(engineer.feature_names, 1):
    print(f"{i:2d}. {feature}")

In [ ]:
# View sample data
print("\n📊 Sample Features (last 5 rows):")
X.tail()

In [ ]:
# Visualize feature correlations
plt.figure(figsize=(15, 12))
correlation_matrix = X.corr()
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title(f'Feature Correlation Matrix - {symbol}', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Model Training

Train ensemble models (Random Forest, XGBoost, LightGBM) with stacking

In [ ]:
# Initialize trainer
trainer = MLTrainer(symbol, lookback_days=730)

# Train models (this may take a few minutes)
print("🚀 Starting training...\n")
metrics = trainer.full_training_pipeline(
    test_size=0.2,
    train_lstm=False,  # Set to True if you have TensorFlow installed
    cross_validate=False,  # Set to True for cross-validation (slower)
    save_models=True
)

In [ ]:
# Display metrics
print("\n📊 Model Performance Metrics:")
print("="*50)

if 'ensemble' in metrics:
    m = metrics['ensemble']
    print(f"Accuracy:  {m['accuracy']:.4f}")
    print(f"Precision: {m['precision']:.4f}")
    print(f"Recall:    {m['recall']:.4f}")
    print(f"F1 Score:  {m['f1']:.4f}")
    print(f"ROC-AUC:   {m['roc_auc']:.4f}")

In [ ]:
# Get feature importance
top_features = trainer.get_feature_importance(top_n=15)

# Visualize feature importance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_names = ['random_forest', 'xgboost', 'lightgbm']

for idx, model_name in enumerate(model_names):
    if model_name in top_features:
        features = top_features[model_name]
        names = [f[0] for f in features]
        importances = [f[1] for f in features]
        
        axes[idx].barh(names, importances, color='steelblue')
        axes[idx].set_xlabel('Importance', fontweight='bold')
        axes[idx].set_title(f'{model_name.upper()}\nTop Features', fontweight='bold')
        axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Making Predictions

Use trained models to predict market direction

In [ ]:
# Load predictor
predictor = MLPredictor(symbol, model_version='v1')

# Get prediction with explanation
prediction = predictor.predict_with_explanation()

# Display prediction
from ML.ml_predictor import print_prediction
print_prediction(prediction)

In [ ]:
# Visualize prediction probabilities
if 'probability' in prediction:
    probs = prediction['probability']
    
    fig, ax = plt.subplots(figsize=(8, 5))
    directions = ['DOWN', 'UP']
    probabilities = [probs['down'], probs['up']]
    colors = ['red', 'green']
    
    bars = ax.bar(directions, probabilities, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    
    # Add percentage labels
    for bar, prob in zip(bars, probabilities):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{prob*100:.1f}%',
                ha='center', va='bottom', fontsize=14, fontweight='bold')
    
    ax.set_ylabel('Probability', fontweight='bold', fontsize=12)
    ax.set_title(f'Market Direction Prediction - {symbol}', fontweight='bold', fontsize=14)
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Using ML Engine

High-level interface for predictions and analysis

In [ ]:
# Initialize ML Engine
engine = MLEngine()

# Single prediction
pred = engine.predict('SPY')
print(f"\n🎯 Prediction: {pred['final_prediction']}")
print(f"🎲 Confidence: {pred['final_confidence']:.2%}")
print(f"⚠️  Risk Level: {pred['risk_level']}")

In [ ]:
# Batch predictions for multiple symbols
symbols = ['SPY', 'QQQ', 'DIA']
predictions = engine.batch_predict(symbols)

# Create summary DataFrame
summary_data = []
for symbol, pred in predictions.items():
    if 'error' not in pred:
        summary_data.append({
            'Symbol': symbol,
            'Prediction': pred['final_prediction'],
            'Confidence': f"{pred['final_confidence']:.2%}",
            'Risk': pred['risk_level']
        })

summary_df = pd.DataFrame(summary_data)
print("\n📊 Batch Predictions:")
display(summary_df)

In [ ]:
# Market regime detection
regimes = {}
for symbol in ['SPY', 'QQQ', 'DIA']:
    regimes[symbol] = engine.get_market_regime(symbol)

print("\n🌐 Market Regimes:")
for symbol, regime in regimes.items():
    emoji = '🐂' if regime == 'BULL' else '🐻' if regime == 'BEAR' else '↔️'
    print(f"  {emoji} {symbol}: {regime}")

## 6. Advanced Analysis

In [ ]:
# Analyze latest features for current prediction
latest_features = engineer.get_latest_features(symbol)

# Get top features by importance
if 'ensemble' in metrics:
    top_feature_names = [f[0] for f in top_features['random_forest'][:10]]
    
    # Create DataFrame of current values for top features
    current_values = {name: latest_features.get(name, 0) for name in top_feature_names}
    
    feature_df = pd.DataFrame([
        {'Feature': name, 'Current Value': f"{value:.4f}"} 
        for name, value in current_values.items()
    ])
    
    print("\n🔍 Current Values of Top Features:")
    display(feature_df)

In [ ]:
# Compare predictions across different symbols
symbols_to_compare = ['SPY', 'QQQ', 'DIA', '^NSEI']
comparison_data = []

for sym in symbols_to_compare:
    try:
        pred = engine.predict(sym)
        if 'error' not in pred:
            comparison_data.append({
                'Symbol': sym,
                'Direction': pred['final_prediction'],
                'Confidence': pred['final_confidence'],
                'Risk': pred['risk_level'],
                'Regime': engine.get_market_regime(sym)
            })
    except:
        pass

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Confidence scores
    colors = ['green' if d == 'UP' else 'red' for d in comparison_df['Direction']]
    ax1.barh(comparison_df['Symbol'], comparison_df['Confidence'], color=colors, alpha=0.7)
    ax1.set_xlabel('Confidence Score', fontweight='bold')
    ax1.set_title('Prediction Confidence by Symbol', fontweight='bold')
    ax1.set_xlim(0, 1)
    
    # Direction distribution
    direction_counts = comparison_df['Direction'].value_counts()
    ax2.pie(direction_counts.values, labels=direction_counts.index, autopct='%1.1f%%',
            colors=['green', 'red'], startangle=90)
    ax2.set_title('Direction Distribution', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Comparison Table:")
    display(comparison_df)

## 7. Save Your Work

In [ ]:
# Export predictions to CSV
if comparison_data:
    comparison_df.to_csv('ml_predictions_export.csv', index=False)
    print("✅ Predictions exported to ml_predictions_export.csv")

## 🎓 Next Steps

1. **Train models for different symbols**: Use `MLTrainer` for any yfinance symbol
2. **Experiment with features**: Modify `feature_engineering.py` to add custom indicators
3. **Tune hyperparameters**: Adjust model parameters in `ml_models.py`
4. **Add LSTM**: Set `train_lstm=True` if you have TensorFlow installed
5. **Build trading strategies**: Use predictions to develop automated trading systems

For more information, see:
- `ML/README.md` - Quick start guide
- `ML/USAGE.md` - Detailed usage examples
- `ML_FOLDER_INFO.md` - Folder structure documentation